# 01 - Universe and data

Bank-level asset quality for the four largest euro-area banking systems,
taken from the **EBA EU-wide Transparency Exercise**. This is supervisory
reporting, published per institution, so every figure here traces to a
regulatory submission rather than to a vendor estimate.

**Why these four.** The intended scope is Italy, Spain, Germany, France and the UK.
UK institutions left the EBA sample after Brexit, so the UK leg is not
addressable from this source. That is a stated gap, not a silent one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)


In [2]:
from src import eba_data as E

panel = E.build_bank_panel()
print("Reporting period:", E.period_label(panel['period'].iloc[0]))
print("Banks in universe:", len(panel))
panel['country_name'].value_counts()

Reporting period: 2025-Q2
Banks in universe: 48


country_name
Germany    17
Italy      11
France     10
Spain      10
Name: count, dtype: int64

## What the exercise actually reports

The exercise is published long-format: one row per bank, period, item code and
dimension combination. Totals are the rows where every breakdown dimension is
zero. `Perf_Status` then carries the split we care about:

| code | meaning |
|---|---|
| 0 | total gross loans |
| 2 | non-performing |
| 12 | performing, but significant increase in credit risk (Stage 2) |

Stage 2 is the one most screens ignore. These loans are performing today, so
they sit outside the NPL ratio entirely -- but they are the population from
which next year's NPLs are drawn. For an origination desk that is the more
useful number, because it is forward-looking.

In [3]:
cols = ['bank','country','loans_total','loans_npl','npl_ratio',
        'stage2_ratio','coverage_ratio','cet1_ratio','net_npl_to_cet1']
panel[cols].head(15).round(1)

,bank,country,loans_total,loans_npl,npl_ratio,stage2_ratio,coverage_ratio,cet1_ratio,net_npl_to_cet1
0,Bpifrance,FR,58597.0,3932.8,6.7,17.9,23.0,25.0,12.8
1,Deutsche Pfandbriefbank AG,DE,35972.3,2021.9,5.6,15.2,35.3,15.3,48.4
2,ATLANTIC LUX HOLDCO S.A R.L.,DE,33279.9,1477.0,4.4,13.5,21.5,21.9,39.7
3,Banca Monte dei Paschi di Siena S.p.A.,IT,83983.5,3064.4,3.6,11.3,46.0,19.6,18.4
4,CASSA CENTRALE BANCA,IT,51903.8,1765.8,3.4,8.2,79.6,27.3,4.0
5,"Banco Santander, S.A.",ES,1073587.2,35924.5,3.3,6.8,39.6,13.0,26.7
6,BANCA POPOLARE DI SONDRIO SOCIETA' PER AZIONI,IT,37608.0,1228.3,3.3,10.0,60.1,15.0,12.4
7,Hamburg Commercial Bank AG,DE,18030.2,580.2,3.2,13.5,26.2,22.1,13.1
8,Landesbank Hessen-Thüringen Girozentrale,DE,125396.8,3919.0,3.1,15.2,29.6,16.3,29.9
9,"Banco Bilbao Vizcaya Argentaria, S.A.",ES,471586.8,14136.2,3.0,6.4,52.6,13.3,13.0


## Sanity checks

A screen built on mis-parsed regulatory data is worse than no screen, so the
aggregates are checked against published EBA figures before anything is built
on top.

In [4]:
agg_npl = 100 * panel['loans_npl'].sum() / panel['loans_total'].sum()
agg_s2  = 100 * panel['loans_stage2'].sum() / panel['loans_total'].sum()
print(f"Weighted NPL ratio    : {agg_npl:.2f}%   (EBA publishes c.2% area)")
print(f"Weighted Stage 2 share: {agg_s2:.2f}%   (EBA publishes high single digit)")
print(f"CET1 range            : {panel['cet1_ratio'].min():.1f}% - {panel['cet1_ratio'].max():.1f}%")
print("\nNulls across derived fields:")
print(panel[cols].isna().sum())

Weighted NPL ratio    : 2.41%   (EBA publishes c.2% area)
Weighted Stage 2 share: 9.57%   (EBA publishes high single digit)
CET1 range            : 12.2% - 43.5%

Nulls across derived fields:
bank               0
country            0
loans_total        0
loans_npl          0
npl_ratio          0
stage2_ratio       0
coverage_ratio     0
cet1_ratio         0
net_npl_to_cet1    0
dtype: int64
